## AI‑Driven Self‑Serve Analytical Intelligence Platform (MVP)

This notebook demonstrates an end‑to‑end prototype of an AI‑driven self‑serve analytics system.
It converts natural language business questions into governed, explainable, and auditable analytical answers.

**Capabilities demonstrated**
- Natural Language → Analytical Intent
- Guideline‑driven Hypothesis Generation
- NL2SQL with Governance Guardrails
- DuckDB Query Execution
- Answer Synthesis with Visualization


### 1. Environment Setup


In [1]:
!pip install duckdb pandas numpy faker python-dateutil
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.9 MB/s eta 0:00:00


### 2. Synthetic Payments Dataset Generation

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker()
np.random.seed(42)

N_ROWS = 50000
START_DATE = datetime.now() - timedelta(days=365)

payment_methods = ["UPI", "CARD", "WALLET"]
statuses = ["initiated", "failed", "cleared"]
regions = ["NORTH", "SOUTH", "EAST", "WEST"]
failure_reasons = ["timeout", "bank_error", "insufficient_funds"]

data = []

for _ in range(N_ROWS):
    date = START_DATE + timedelta(days=random.randint(0, 364))
    status = random.choices(statuses, weights=[0.1, 0.2, 0.7])[0]

    data.append({
        "transaction_id": fake.uuid4(),
        "transaction_date": date.date(),
        "payment_method": random.choices(payment_methods, weights=[0.6, 0.25, 0.15])[0],
        "transaction_status": status,
        "net_settlement_amount": round(random.uniform(10, 5000), 2) if status == "cleared" else 0,
        "region": random.choice(regions),
        "failure_reason": random.choice(failure_reasons) if status == "failed" else None,
        "customer_id": fake.uuid4(),
        "country": "IN"
    })

df_transactions = pd.DataFrame(data)
df_transactions.head()


,transaction_id,transaction_date,payment_method,transaction_status,net_settlement_amount,region,failure_reason,customer_id,country
0,d038eb69-a2e6-4083-8e79-049ebd4210c7,2025-05-14,UPI,cleared,1011.46,WEST,None,764ffa73-d1e7-4f51-9702-af386c4aa9e3,IN
1,d856b692-f3f7-4b25-ac4b-e27e7b7589ad,2026-02-12,UPI,cleared,2117.32,NORTH,None,65e0db6e-f886-4c2a-97c2-c091363d2c7e,IN
2,7aecc1fb-97f7-4611-b3d2-36a63c472289,2026-02-10,UPI,failed,0.00,SOUTH,bank_error,d6c71eee-0644-4a84-983f-887495130728,IN
3,e710c5b7-7078-4f2f-a557-93640772dc23,2025-04-22,UPI,cleared,1932.03,WEST,None,2cd598ec-e577-4a57-9d68-f89523efa53f,IN
4,a1f9ffe3-1238-4eb2-840f-a05784f9088d,2025-10-01,CARD,failed,0.00,SOUTH,insufficient_funds,0e030621-0cd9-4439-84a2-549e2c2ee08a,IN


### 3. DuckDB In‑Memory Analytics Engine

In [3]:
import duckdb

con = duckdb.connect(database=":memory:")
con.execute("CREATE TABLE transactions AS SELECT * FROM df_transactions")

### 4. Business Glossary & Governance Rules

In [4]:
BUSINESS_GLOSSARY = {
    "revenue": {
        "expression": "SUM(net_settlement_amount)",
        "mandatory_filters": {"transaction_status": "cleared"},
        "description": "Net settled revenue from cleared transactions"
    },
    "failed_transactions": {
        "expression": "COUNT(transaction_id)",
        "mandatory_filters": {"transaction_status": "failed"},
        "description": "Total failed transactions"
    }
}

GOVERNANCE = {
    "pii_columns": ["customer_id"],
    "row_level_filters": {"country": "IN"}
}

### 5. Schema Catalog & Metadata Store

In [5]:
SCHEMA_CATALOG = {
    "transactions": {
        "columns": {
            "transaction_date": "Date of transaction",
            "payment_method": "Payment mode",
            "transaction_status": "Transaction lifecycle status",
            "net_settlement_amount": "Final settled amount",
            "region": "Geographical region",
            "failure_reason": "Reason for transaction failure"
        }
    }
}

 This will later be embedded for semantic retrieval.

### 6. Query Understanding Layer

In [6]:
def understand_query(question: str):
    """
    Simplified LLM‑style intent extraction (stub for MVP)
    """
    intent = {
        "metric": "revenue" if "revenue" in question.lower() else None,
        "time_range": "last_month" if "last month" in question.lower() else None,
        "dimensions": ["region"] if "region" in question.lower() else [],
        "intent_type": "temporal_comparison" if "change" in question.lower() else "aggregate_metric"
    }
    return intent

We can later replace this with a real LLM call.

### 7. Hypothesis Engine (Guideline‑Driven)

In [7]:
ALLOWED_HYPOTHESIS_TYPES = [
    "aggregate_metric",
    "temporal_comparison",
    "dimensional_segmentation",
    "driver_analysis",
    "cohort_analysis",
    "anomaly_explanation"
]

def generate_hypotheses(intent):
    hypotheses = []

    if intent["intent_type"] == "aggregate_metric":
        hypotheses.append({
            "type": "aggregate_metric",
            "metric": intent["metric"]
        })

    if intent["intent_type"] == "temporal_comparison":
        hypotheses.append({
            "type": "temporal_comparison",
            "metric": intent["metric"],
            "comparison": "MoM"
        })

    return hypotheses

### 8. NL2SQL Engine

In [8]:
def generate_sql(hypothesis):
    metric_def = BUSINESS_GLOSSARY[hypothesis["metric"]]

    sql = f"""
    SELECT
        strftime(transaction_date, '%Y-%m') AS month,
        {metric_def['expression']} AS value
    FROM transactions
    WHERE transaction_status = 'cleared'
      AND country = 'IN'
    GROUP BY month
    ORDER BY month
    """
    return sql.strip()

### 9. Query Execution

In [9]:
def execute_query(sql):
    return con.execute(sql).df()

### 10. Answer Synthesis & Visualization

In [10]:
def synthesize_answer(df):
    delta = df["value"].pct_change().iloc[-1] * 100
    return f"Revenue changed by {delta:.2f}% month over month."

### 11. End‑to‑End Demo

In [11]:
question = "How did revenue change last month?"

intent = understand_query(question)
hypotheses = generate_hypotheses(intent)
sql = generate_sql(hypotheses[0])
result = execute_query(sql)

print("SQL Generated:")
print(sql)
print("\nAnswer:")
print(synthesize_answer(result))
result.head()

SQL Generated:
SELECT
        strftime(transaction_date, '%Y-%m') AS month,
        SUM(net_settlement_amount) AS value
    FROM transactions
    WHERE transaction_status = 'cleared'
      AND country = 'IN'
    GROUP BY month
    ORDER BY month

Answer:
Revenue changed by -63.03% month over month.


,month,value
0,2025-04,4238284.39
1,2025-05,7760396.16
2,2025-06,7026920.56
3,2025-07,7418530.09
4,2025-08,7426177.19
